[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/certified-journeys/certified-journeys.github.io/blob/main/courses/sodacore-certified/notebooks/day-04-threshold-freshness-checks.ipynb#scrollTo=a1b2c3d4)

---
# Day 4 · Threshold Checks — Valid Ranges, Freshness, and Volume Anomaly Detection
**certified-journeys / sodacore-certified** · Day 4 · Practice

> **Goal for today:** Write range checks that fail on out-of-range data, configure freshness thresholds with warn/fail levels, and detect volume anomalies by comparing row counts to a baseline — all using the Soda Core Python API against a local DuckDB database.


In [ ]:
%pip install -q soda-core-duckdb


## Step 1 · Setting Up: Synthetic Data and Soda Config

Soda Core works by reading a **configuration YAML** (data source credentials) and a **checks YAML** (the quality rules). For DuckDB, the only credential needed is the path to the `.duckdb` file.

| File | Purpose |
|---|---|
| `configuration.yml` | Data source type + connection params |
| `checks.yml` | Quality rules written in SodaCL |

We use `pathlib` + `tempfile` to keep everything self-contained — no permanent files are written to the notebook directory.

> **Production equivalent:** Replace the DuckDB path with a Snowflake/BigQuery/Redshift URI in `configuration.yml`. The checks YAML stays the same.


In [ ]:
import duckdb
import tempfile
import pathlib
import datetime

# Create a temp directory — all files (DB + YAMLs) live here
tmpdir = pathlib.Path(tempfile.mkdtemp())
db_path = str(tmpdir / "orders.duckdb")

conn = duckdb.connect(db_path)
conn.execute("""
    CREATE TABLE orders (
        id          INTEGER,
        customer_id INTEGER,
        revenue     DOUBLE,
        updated_at  TIMESTAMP
    )
""")

# Insert a mix of VALID and OUT-OF-RANGE revenue rows
now = datetime.datetime.now()
conn.execute("""
    INSERT INTO orders VALUES
        (1,  101, 150.00,  CURRENT_TIMESTAMP),
        (2,  102, 750.50,  CURRENT_TIMESTAMP),
        (3,  103, -25.00,  CURRENT_TIMESTAMP),   -- INVALID: negative revenue
        (4,  104, 1200.00, CURRENT_TIMESTAMP),   -- INVALID: above max 1000
        (5,  105, 0.00,    CURRENT_TIMESTAMP),
        (6,  106, 999.99,  CURRENT_TIMESTAMP)
""")
conn.close()

print(f"Database created at: {db_path}")

# Write configuration.yml — points Soda at the DuckDB file
config_yml = f"""
data_sources:
  ordersdb:
    type: duckdb
    path: "{db_path}"
"""

config_path = tmpdir / "configuration.yml"
config_path.write_text(config_yml)
print("configuration.yml written.")


### What just happened?
- A fresh `.duckdb` file was created with **6 rows**: 2 are invalid (negative revenue, revenue > 1000).
- **`configuration.yml`** tells Soda the data source name (`ordersdb`) and its type/path — this is the only file you change when moving between environments.
- All paths are in a `tempdir` so nothing persists after the Colab session.
- `conn.close()` releases the DuckDB write lock before Soda opens its own connection.


## Step 2 · Range Checks: min >= 0 and max <= 1000

SodaCL range checks use the `min()` and `max()` metric expressions with threshold syntax:

```yaml
checks for orders:
  - min(revenue) >= 0       # FAIL if any revenue is negative
  - max(revenue) <= 1000    # FAIL if any revenue exceeds 1000
```

The threshold is evaluated as: *"the metric value must satisfy this condition"*. If the condition is false, the check **fails**.

| SodaCL syntax | Meaning |
|---|---|
| `min(col) >= 0` | Minimum value across all rows must be ≥ 0 |
| `max(col) <= 1000` | Maximum value across all rows must be ≤ 1000 |
| `row_count > 0` | Table must be non-empty |


In [ ]:
from soda.scan import Scan

# Write the checks YAML with range rules
range_checks_yml = """
checks for orders:
  - row_count > 0
  - min(revenue) >= 0:
      name: revenue_no_negatives
  - max(revenue) <= 1000:
      name: revenue_within_max
"""

checks_path = tmpdir / "range_checks.yml"
checks_path.write_text(range_checks_yml)

# Run the scan
scan = Scan()
scan.set_data_source_name("ordersdb")
scan.add_configuration_yaml_file(str(config_path))
scan.add_sodacl_yaml_file(str(checks_path))
scan.execute()

# Print results
print(scan.get_logs_text())
print("\n--- Check outcomes ---")
for check in scan.get_checks():
    print(f"  [{check.outcome}] {check.name}")

# Exit code: 0 = all pass, 2 = at least one FAIL, 1 = warnings
exit_code = scan.get_exit_code()
print(f"\nScan exit code: {exit_code}  (2 = FAIL)")


### What just happened?
- Both range checks **failed** because the table contains `revenue = -25.00` (violates `min >= 0`) and `revenue = 1200.00` (violates `max <= 1000`).
- **`scan.get_checks()`** returns a list of check result objects; each has `.outcome` (`pass`, `warn`, or `fail`) and `.name`.
- **Exit code 2** means at least one check failed — in a CI pipeline you'd raise on this code to block a deploy.
- `row_count > 0` passed because the table is non-empty.


## Step 3 · Freshness Checks with warn and fail Thresholds

Freshness checks measure the age of the **most recent row** in a timestamp column and compare it to a threshold.

```yaml
checks for orders:
  - freshness(updated_at) < 24h:
      warn: when > 12h
      fail: when > 24h
```

Soda evaluates freshness as `NOW() - MAX(updated_at)`. If the gap exceeds the warn threshold, the check outcome is `warn`; if it exceeds the fail threshold, the outcome is `fail`.

| Age of newest row | Outcome |
|---|---|
| < 12 h | pass |
| 12 h – 24 h | warn |
| > 24 h | fail |

> **Why freshness matters:** A stale table that looks healthy on row count or range checks can still silently feed dashboards with outdated data.


In [ ]:
# ── Helper: create a fresh DB with rows at a controlled age ──────────────────
def build_freshness_db(hours_old: float) -> str:
    """Return path to a .duckdb file whose newest row is `hours_old` hours old."""
    path = str(tmpdir / f"freshness_{int(hours_old*10):04d}.duckdb")
    ts = datetime.datetime.now() - datetime.timedelta(hours=hours_old)
    c = duckdb.connect(path)
    c.execute("CREATE TABLE orders (id INTEGER, updated_at TIMESTAMP)")
    c.execute(f"INSERT INTO orders VALUES (1, TIMESTAMP '{ts.strftime('%Y-%m-%d %H:%M:%S')}')")
    c.close()
    return path


def run_freshness_scan(db_path: str, label: str) -> None:
    """Run a freshness scan and print the outcome."""
    cfg_yml = f"""
data_sources:
  ordersdb:
    type: duckdb
    path: "{db_path}"
"""
    chk_yml = """
checks for orders:
  - freshness(updated_at) < 24h:
      name: data_freshness
      warn: when > 12h
      fail: when > 24h
"""
    cfg_p = tmpdir / f"cfg_{label}.yml"
    chk_p = tmpdir / f"chk_{label}.yml"
    cfg_p.write_text(cfg_yml)
    chk_p.write_text(chk_yml)

    s = Scan()
    s.set_data_source_name("ordersdb")
    s.add_configuration_yaml_file(str(cfg_p))
    s.add_sodacl_yaml_file(str(chk_p))
    s.execute()

    for check in s.get_checks():
        print(f"  [{label:10s}] outcome={check.outcome}  name={check.name}")


# Scenario A: data is 5 hours old → should PASS
run_freshness_scan(build_freshness_db(5),  "5h_old")

# Scenario B: data is 15 hours old → should WARN
run_freshness_scan(build_freshness_db(15), "15h_old")

# Scenario C: data is 30 hours old → should FAIL
run_freshness_scan(build_freshness_db(30), "30h_old")


### What just happened?
- Three databases were created with rows timestamped 5 h, 15 h, and 30 h in the past.
- **5 h** → `pass` (below warn threshold of 12 h).
- **15 h** → `warn` (between 12 h warn and 24 h fail).
- **30 h** → `fail` (beyond 24 h fail threshold).
- Soda computes `NOW() - MAX(updated_at)` automatically — no SQL needed in the checks file.


## Step 4 · Volume Anomaly Detection: Row Count vs Baseline

Volume checks guard against **silent data loss or duplication** — situations where data arrives but is truncated, or where a reprocessing bug inserts duplicate rows.

Soda supports two patterns:

| Pattern | SodaCL syntax | Use case |
|---|---|---|
| Hard threshold | `row_count between 900 and 1100` | Known stable volume |
| Change relative to previous scan | `change for row_count < 20%` | Trending / growing tables |

For the hard-threshold pattern, you establish a **baseline** (e.g. from yesterday's row count) and encode it as the expected range.

> **Production tip:** Store baselines in Soda Cloud or in a metadata table; re-read them at scan time to make the threshold dynamic.


In [ ]:
# ── Build three volume scenarios ─────────────────────────────────────────────
def build_volume_db(n_rows: int, name: str) -> str:
    """Create a .duckdb file with n_rows rows in `orders`."""
    path = str(tmpdir / f"vol_{name}.duckdb")
    c = duckdb.connect(path)
    c.execute("CREATE TABLE orders (id INTEGER, amount DOUBLE)")
    # Generate n_rows using DuckDB's range() — much faster than Python loops
    c.execute(f"""
        INSERT INTO orders
        SELECT i, round(random() * 100 + 1, 2)
        FROM range(1, {n_rows + 1}) t(i)
    """)
    c.close()
    return path


def run_volume_scan(db_path: str, label: str,
                    low: int = 900, high: int = 1100) -> None:
    """Run a volume check asserting row_count between low and high."""
    cfg_yml = f"""
data_sources:
  ordersdb:
    type: duckdb
    path: "{db_path}"
"""
    chk_yml = f"""
checks for orders:
  - row_count between {low} and {high}:
      name: volume_within_baseline
      fail: when not between {low} and {high}
      warn: when not between {int(low * 0.95)} and {int(high * 1.05)}
"""
    cfg_p = tmpdir / f"cfg_vol_{label}.yml"
    chk_p = tmpdir / f"chk_vol_{label}.yml"
    cfg_p.write_text(cfg_yml)
    chk_p.write_text(chk_yml)

    s = Scan()
    s.set_data_source_name("ordersdb")
    s.add_configuration_yaml_file(str(cfg_p))
    s.add_sodacl_yaml_file(str(chk_p))
    s.execute()

    # Read the actual row count for reporting
    c = duckdb.connect(db_path, read_only=True)
    actual = c.execute("SELECT COUNT(*) FROM orders").fetchone()[0]
    c.close()

    for check in s.get_checks():
        print(f"  [{label:12s}] rows={actual:5d}  outcome={check.outcome}  "
              f"baseline=[{low}, {high}]")


# Scenario 1: 1 000 rows — within baseline [900, 1100] → PASS
run_volume_scan(build_volume_db(1000, "normal"),  "normal")

# Scenario 2: 400 rows — far below baseline → FAIL (data loss)
run_volume_scan(build_volume_db(400,  "low"),     "low_volume")

# Scenario 3: 2 500 rows — far above baseline → FAIL (possible duplicate ingestion)
run_volume_scan(build_volume_db(2500, "high"),    "high_volume")


### What just happened?
- **Normal load (1 000 rows)** passed because the count sits within the [900, 1100] baseline range.
- **Low volume (400 rows)** failed — a 60% drop would indicate a truncation or pipeline failure upstream.
- **High volume (2 500 rows)** failed — a 2.5× spike often indicates duplicate processing.
- **The inner warn band** (5% tighter than the fail band) provides an early-warning buffer before a hard fail.


## Step 5 · Reading Detailed Check Results Programmatically

Beyond `scan.get_logs_text()`, Soda exposes structured check results that are easy to parse:

| Method | Returns |
|---|---|
| `scan.get_checks()` | List of `SodaCheck` objects |
| `check.outcome` | `"pass"`, `"warn"`, or `"fail"` |
| `check.name` | Human-readable check name |
| `scan.get_exit_code()` | `0`=all pass, `1`=warns, `2`=fails |
| `scan.has_check_failures()` | `True` if any check failed |

This structured access is what makes Soda useful inside Python pipelines — you can branch on outcomes without parsing log strings.


In [ ]:
# Full scan combining ALL check types from today
# Re-use the original orders.duckdb (has bad range data + current timestamps)

combined_checks_yml = """
checks for orders:
  - row_count between 1 and 100:
      name: volume_check
  - min(revenue) >= 0:
      name: no_negative_revenue
  - max(revenue) <= 1000:
      name: revenue_within_max
  - freshness(updated_at) < 24h:
      name: data_freshness
      warn: when > 12h
      fail: when > 24h
"""

combined_path = tmpdir / "combined_checks.yml"
combined_path.write_text(combined_checks_yml)

scan = Scan()
scan.set_data_source_name("ordersdb")
scan.add_configuration_yaml_file(str(config_path))   # config_path from Step 1
scan.add_sodacl_yaml_file(str(combined_path))
scan.execute()

# Structured summary
print("=" * 50)
print("COMBINED SCAN RESULTS")
print("=" * 50)
for check in scan.get_checks():
    icon = {"pass": "✓", "warn": "⚠", "fail": "✗"}.get(check.outcome, "?")
    print(f"  {icon} [{check.outcome.upper():4s}] {check.name}")

print()
print(f"Exit code      : {scan.get_exit_code()}")
print(f"Has failures   : {scan.has_check_failures()}")

# Example: raise in a pipeline when any check fails
if scan.has_check_failures():
    print("\nPipeline gate: BLOCKED — one or more checks failed.")
    # In production: raise RuntimeError("Soda checks failed")
else:
    print("\nPipeline gate: OPEN — all checks passed.")


### What just happened?
- All four check types ran in a **single scan** against the same data source.
- **`scan.has_check_failures()`** gives a boolean gate — the cleanest way to raise or block downstream tasks.
- The structured `get_checks()` loop produces a machine-readable summary table, suitable for Slack notifications or Airflow task annotations.
- Freshness **passed** because the data was inserted moments ago; range checks failed due to the intentional bad data from Step 1.


In [ ]:
# Challenge: Volume + Freshness + Range combo
#
# 1. Create a new DuckDB table called `transactions` with columns:
#      id INTEGER, amount DOUBLE, created_at TIMESTAMP
# 2. Insert 50 rows where:
#    - 48 rows have valid amount (10–500) and created_at = NOW()
#    - 2 rows have amount = -99 (invalid)
# 3. Write a SodaCL checks file that:
#    - Asserts row_count between 40 and 60
#    - Asserts min(amount) >= 0  (should FAIL)
#    - Asserts freshness(created_at) < 1h (should PASS)
# 4. Run the scan and print each check's outcome
#
# Expected: row_count → pass, min(amount) → fail, freshness → pass

# Your solution here
# tx_db_path = str(tmpdir / "transactions.duckdb")
# ...


---
## Day 4 key concepts recap

| Concept | What to remember |
|---|---|
| Range checks | `min(col) >= N` / `max(col) <= N` — metric must satisfy the condition |
| Freshness check | Compares `NOW() - MAX(timestamp_col)` to warn/fail thresholds |
| Warn vs fail thresholds | `warn: when > Xh` / `fail: when > Yh` inside the same check block |
| Volume baseline | `row_count between low and high` detects silent data loss or duplication |
| Exit codes | `0` = all pass, `1` = warnings, `2` = at least one fail |
| Pipeline gate | `scan.has_check_failures()` → raise to block downstream tasks |

> **Tip:** Always set both a `warn` and a `fail` threshold on freshness. The warn level buys the on-call engineer time to investigate before data consumers notice.

---
## What's next
**Day 5** → Custom SQL checks and metric expressions — write arbitrary SQL to identify failed rows and define your own computed metrics.

Mark Day 4 complete in your [tracker](../index.html).
